# Smart Factory Energy Analytics — Synthetic Demonstration

All observations are synthetic. This notebook is a reproducible portfolio demonstration, **not** the original company analysis.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

In [ ]:
path = Path("../data/synthetic_factory_energy.csv")
if not path.exists():
    path = Path("data/synthetic_factory_energy.csv")
df = pd.read_csv(path, parse_dates=["date"])
df.head()

## Explore associations
Correlations describe associations, not causality.

In [ ]:
df.select_dtypes("number").corr()["total_energy_kwh"].sort_values(ascending=False)

In [ ]:
df.plot(x="date", y="total_energy_kwh", figsize=(10, 4), title="Synthetic daily energy use")
plt.show()

## Cluster operating modes
Standardise features so their different units do not dominate distance-based clustering.

In [ ]:
features = ["ambient_temperature_c", "ambient_humidity_pct", "finished_goods_units", "machine_runtime_hours", "cooling_energy_kwh", "compressor_energy_kwh", "injection_energy_kwh"]
X = StandardScaler().fit_transform(df[features].fillna(df[features].median()))
scores = {k: silhouette_score(X, KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(X)) for k in range(2, 6)}
scores

In [ ]:
best_k = max(scores, key=scores.get)
labels = KMeans(n_clusters=best_k, n_init=10, random_state=42).fit_predict(X)
df.assign(cluster=labels).groupby("cluster")[["total_energy_kwh", "finished_goods_units", "cooling_energy_kwh"]].mean().round(1)

In [ ]:
coords = PCA(n_components=2).fit_transform(X)
plt.scatter(coords[:, 0], coords[:, 1], c=labels, s=18)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Synthetic operating modes")
plt.show()

## Interpretation
Use this demonstration to discuss exploratory data analysis and clustering. Do not present synthetic metrics as real-world factory outcomes or as reproducing the dissertation's autoencoder results.